<a href="https://colab.research.google.com/github/Sarahnjunge/Enveda_Casmi_2026/blob/main/coconut_candidates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

enveda_casmi26_molecule_id_mass_spectra_path = kagglehub.competition_download('enveda-CASMI26-molecule-id-mass-spectra')
sarahnjunge_coconut_candidates_path = kagglehub.notebook_output_download('sarahnjunge/coconut-candidates')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/__notebook__.ipynb
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/__output__.json
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/__results__.html
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/custom.css
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/ref_mat.npz
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/ref_meta.parquet
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/submission.csv


In [ ]:
# Enveda CASMI 2026 - baseline: mass-filtered spectral library search
# Designed to run as a Kaggle notebook (paste cells split by "# %%") or as a script.
#
# Idea
#   1. Turn every spectrum into a sparse vector: [fragment-m/z bins | neutral-loss bins].
#      Cosine on this vector ~ a cheap "modified cosine" (loss block lets analogs match).
#   2. For each test molecule, take its neutral mass (from precursor + adduct), pull all
#      reference spectra within a growing mass window, and score each reference structure
#      = mean over the molecule's spectra of (best cosine over that structure's spectra).
#   3. Rank by (mass tier, score) and emit the top 25 SMILES.
#
# Env vars: DATA_DIR (folder with train/test/sample_submission), OUT_DIR, QUICK=1 (subsample train)

# %% imports / config
import os, re, glob, time, sys
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import scipy.sparse as sp

def _find_dir():
    if os.environ.get("DATA_DIR"):
        return os.environ["DATA_DIR"]
    hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
    return os.path.dirname(hits[0]) if hits else "."

DATA_DIR = _find_dir()
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
QUICK = int(os.environ.get("QUICK", "0"))

BIN = 0.02            # m/z bin width (Da)
MAX_MZ = 1300.0
DIM = int(MAX_MZ / BIN) + 4
TOPN = 64             # peaks kept per spectrum
MIN_REL = 0.01        # relative intensity floor
TOPK = 25
TIERS_DA = None       # filled per molecule (first tier is ppm based)
PPM = 15.0
TIER_ABS = [0.05, 0.5, 3.0, 15.0]   # widening windows (Da) used to fill up to 25 candidates
MAX_CAND_SPECTRA = 150_000
MAX_PPM_ERR_TRAIN = 20.0

PROTON, ELECTRON = 1.007276, 0.000549
H2O, NH4, NA, K, CL, FA = 18.010565, 18.033823, 22.989221, 38.963158, 34.969402, 46.005479
ADDUCT_DELTA = {   # ion m/z = M + delta
    "[M+H]+": PROTON, "[M+NH4]+": NH4, "[M-H2O+H]+": PROTON - H2O,
    "[M-2H2O+H]+": PROTON - 2 * H2O, "[M+Na]+": NA, "[M+K]+": K,
    "[M-H]-": -PROTON, "[M-H2O-H]-": -PROTON - H2O,
    "[M+CH2O2-H]-": FA - PROTON, "[M+Cl]-": CL,
}
EL = {"H": 1.00782503, "C": 12.0, "N": 14.0030740, "O": 15.9949146, "S": 31.9720707,
      "P": 30.9737620, "F": 18.9984032, "Cl": 34.9688527, "Br": 78.9183376, "I": 126.904473,
      "Na": 22.9897693, "K": 38.9637069, "Si": 27.9769265, "B": 11.0093054, "Se": 79.9165218,
      "Fe": 55.9349375, "Mg": 23.9850417, "Ca": 39.9625912, "Zn": 63.9291422, "Cu": 62.9295975,
      "As": 74.9215965, "Li": 7.01600455, "Al": 26.9815386, "Mn": 54.9380451, "Co": 58.9331950,
      "Ni": 57.9353429, "Sn": 119.9022016, "Pt": 194.9647917, "Hg": 201.970643, "Sb": 120.9038157}

def formula_mass(f):
    if not isinstance(f, str) or not f:
        return np.nan
    tot = 0.0
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", f):
        if el not in EL:
            return np.nan
        tot += EL[el] * (int(n) if n else 1)
    return tot

# %% spectrum -> sparse vector
def spectra_to_csr(mz_col, int_col, prec):
    """mz_col / int_col: pyarrow ListArray; prec: float array. Fully vectorised."""
    n = len(prec)
    off = mz_col.offsets.to_numpy()
    off = off - off[0]
    lens = np.diff(off)
    mz = mz_col.flatten().to_numpy(zero_copy_only=False).astype(np.float64)
    it = int_col.flatten().to_numpy(zero_copy_only=False).astype(np.float64)
    sid = np.repeat(np.arange(n), lens)
    keep = (mz < prec[sid] - 0.5) & (mz >= 30.0) & (it >= MIN_REL)
    sid, mz, it = sid[keep], mz[keep], it[keep]
    order = np.lexsort((-it, sid))                # by spectrum, then intensity desc
    sid, mz, it = sid[order], mz[order], it[order]
    first = np.searchsorted(sid, np.arange(n))
    rank = np.arange(len(sid)) - first[sid]
    k = rank < TOPN
    sid, mz, it = sid[k], mz[k], it[k]
    norm = np.sqrt(np.bincount(sid, weights=it, minlength=n))   # = ||sqrt(I)||
    val = np.sqrt(it) / norm[sid] * np.sqrt(0.5)                # each block unit-norm * sqrt(.5)
    fb = np.rint(mz / BIN).astype(np.int64)
    lb = np.rint((prec[sid] - mz) / BIN).astype(np.int64)
    ok = (fb < DIM) & (lb < DIM) & (lb >= 0)
    sid, fb, lb, val = sid[ok], fb[ok], lb[ok], val[ok]
    rows = np.concatenate([sid, sid])
    cols = np.concatenate([fb, lb + DIM])
    data = np.concatenate([val, val]).astype(np.float32)
    return sp.csr_matrix((data, (rows, cols)), shape=(n, 2 * DIM), dtype=np.float32)

def read_spectra(path, meta_cols, batch=200_000, max_rows=None):
    pf = pq.ParquetFile(path)
    spec_cols = ["ms2_mzs", "ms2_normalized_intensities"]
    metas, mats, seen = [], [], 0
    for b in pf.iter_batches(batch_size=batch, columns=meta_cols + spec_cols):
        t = pa.Table.from_batches([b])
        meta = t.select(meta_cols).to_pandas()
        prec = meta["precursor_mz"].to_numpy(float)
        mats.append(spectra_to_csr(b.column("ms2_mzs"), b.column("ms2_normalized_intensities"), prec))
        metas.append(meta)
        seen += len(meta)
        print(f"  read {seen:,} spectra", flush=True)
        if max_rows and seen >= max_rows:
            break
    return pd.concat(metas, ignore_index=True), sp.vstack(mats, format="csr")

# %% build / load reference
TRAIN_META = ["inchikey14", "normalized_smiles", "molecular_formula", "ionization_mode",
              "precursor_mz", "precursor_error_ppm", "ingest_lib", "adduct"]
TEST_META = ["molecule_id", "spectrum_id", "adduct", "ionization_mode", "precursor_mz",
             "base_peak_intensity"]

def build_reference():
    cache_meta = os.path.join(OUT_DIR, "ref_meta.parquet")
    cache_mat = os.path.join(OUT_DIR, "ref_mat.npz")
    if os.path.exists(cache_meta) and os.path.exists(cache_mat) and not QUICK:
        print("loading cached reference")
        return pd.read_parquet(cache_meta), sp.load_npz(cache_mat).tocsr()
    t0 = time.time()
    meta, mat = read_spectra(os.path.join(DATA_DIR, "train.parquet"), TRAIN_META,
                             max_rows=300_000 if QUICK else None)
    smass = meta["molecular_formula"].map(
        {f: formula_mass(f) for f in meta["molecular_formula"].dropna().unique()})
    good = (smass.notna() & meta["inchikey14"].notna() & meta["normalized_smiles"].notna()
            & meta["ionization_mode"].isin(["positive", "negative"])
            & ~(meta["precursor_error_ppm"].abs() > MAX_PPM_ERR_TRAIN)
            & (np.diff(mat.indptr) > 2)).to_numpy()
    meta = meta[good].reset_index(drop=True)
    meta["struct_mass"] = smass[good].to_numpy()
    mat = mat[good]
    meta.to_parquet(cache_meta)
    if not QUICK:
        sp.save_npz(cache_mat, mat)
    print(f"reference: {len(meta):,} spectra, {meta.inchikey14.nunique():,} structures, "
          f"{time.time()-t0:.0f}s")
    return meta, mat

def make_ref(meta, mat):
    sid, uniq = pd.factorize(meta["inchikey14"])
    first = pd.Series(np.arange(len(meta))).groupby(sid).first().to_numpy()
    R = dict(
        mat=mat, sid=sid.astype(np.int32),
        smiles=meta["normalized_smiles"].to_numpy()[first],
        smass=meta["struct_mass"].to_numpy()[first],
        ik=np.asarray(uniq),
        mode=(meta["ionization_mode"] == "positive").to_numpy(),
        lib=meta["ingest_lib"].to_numpy(),
    )
    R["mass"] = R["smass"][R["sid"]]
    R["order"] = np.argsort(R["mass"], kind="stable")
    R["mass_sorted"] = R["mass"][R["order"]]
    return R

# %% search
SPREAD = None
def _spread():
    global SPREAD
    if SPREAD is None:
        SPREAD = sp.diags([0.5, 1.0, 0.5], [-1, 0, 1], shape=(2 * DIM, 2 * DIM), format="csr",
                          dtype=np.float32)
    return SPREAD

def neutral_mass(meta):
    d = meta["adduct"].map(ADDUCT_DELTA)
    return (meta["precursor_mz"] - d).to_numpy(float)

def search_molecule(T, tmode, M, R, ref_ok):
    tols = np.array([PPM * 1e-6 * M] + TIER_ABS)
    idx = None
    for tol in tols:
        lo = np.searchsorted(R["mass_sorted"], M - tol, "left")
        hi = np.searchsorted(R["mass_sorted"], M + tol, "right")
        idx = R["order"][lo:hi]
        idx = idx[ref_ok[idx]]
        if len(np.unique(R["sid"][idx])) >= TOPK:
            break
    if len(idx) == 0:
        return []
    if len(idx) > MAX_CAND_SPECTRA:
        idx = idx[np.argsort(np.abs(R["mass"][idx] - M))[:MAX_CAND_SPECTRA]]
    S = (T @ R["mat"][idx].T).toarray()
    S *= (tmode[:, None] == R["mode"][idx][None, :])          # polarity must match
    sc = R["sid"][idx]
    o = np.argsort(sc, kind="stable")
    S, sc = S[:, o], sc[o]
    starts = np.r_[0, np.flatnonzero(np.diff(sc)) + 1]
    score = np.maximum.reduceat(S, starts, axis=1).mean(0)
    usid = sc[starts]
    dm = np.abs(R["smass"][usid] - M)
    tier = (dm[:, None] > tols[None, :]).sum(1)
    top = usid[np.lexsort((-score, tier))[:TOPK]]
    return list(top)

def run_search(Tmeta, T, R, ref_ok):
    """Tmeta needs molecule_id, ionization_mode, neutral_mass. T: csr rows aligned to Tmeta."""
    Ts = (T @ _spread()).tocsr()
    tmode_all = (Tmeta["ionization_mode"] == "positive").to_numpy()
    nm = Tmeta["neutral_mass"].to_numpy(float)
    out = {}
    groups = Tmeta.groupby("molecule_id").indices
    for i, (mol, rows) in enumerate(groups.items()):
        M = np.nanmedian(nm[rows])
        out[mol] = run = search_molecule(Ts[rows], tmode_all[rows], M, R, ref_ok) if np.isfinite(M) else []
        if i % 50 == 0:
            print(f"  searched {i}/{len(groups)} molecules", flush=True)
    return out

# %% validation on enveda-np-examples (same instrument/pipeline as test)
def validate(meta, mat, R):
    is_np = (R["lib"] == "enveda-np-examples")
    if is_np.sum() == 0:
        print("no enveda-np-examples rows found; skipping validation"); return
    rows = np.flatnonzero(is_np)
    Tmeta = pd.DataFrame({
        "molecule_id": R["ik"][R["sid"][rows]],
        "ionization_mode": np.where(R["mode"][rows], "positive", "negative"),
        "neutral_mass": np.where(
            meta["adduct"].iloc[rows].isin(ADDUCT_DELTA).to_numpy(),
            neutral_mass(meta.iloc[rows]), R["mass"][rows]),
    })
    T = mat[rows]
    np_structs = np.unique(R["sid"][rows])
    for name, ref_ok in [
        ("class-1-like (np-examples spectra removed, other libs kept)", ~is_np),
        ("hard (structure fully removed from reference)", ~np.isin(R["sid"], np_structs)),
    ]:
        res = run_search(Tmeta, T, R, ref_ok)
        hits = {1: 0, 5: 0, 25: 0}
        for mol, top in res.items():
            iks = [R["ik"][s] for s in top]
            for k in hits:
                hits[k] += mol in iks[:k]
        n = len(res)
        print(f"[{name}] molecules={n}  " + "  ".join(f"top{k}={v/n:.3f}" for k, v in hits.items()))

# %% main
def main():
    meta, mat = build_reference()
    R = make_ref(meta, mat)
    if os.environ.get("VALIDATE", "1") == "1":
        validate(meta, mat, R)
    print("loading test")
    tm, T = read_spectra(os.path.join(DATA_DIR, "test.parquet"), TEST_META)
    unk = set(tm["adduct"]) - set(ADDUCT_DELTA)
    if unk:
        print("WARNING unknown adducts:", unk)
    tm["neutral_mass"] = neutral_mass(tm)
    res = run_search(tm, T, R, np.ones(len(R["mass"]), bool))
    sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
    smiles = []
    for mol in sub["molecule_id"]:
        s = [R["smiles"][i] for i in res.get(mol, [])]
        s += ["CCO"] * (TOPK - len(s))
        smiles.append(";".join(s[:TOPK]))
    sub["smiles"] = smiles
    path = os.path.join(OUT_DIR, "submission.csv")
    sub.to_csv(path, index=False)
    print("wrote", path)

if __name__ == "__main__":
    main()

  read 200,000 spectra
  read 400,000 spectra
  read 600,000 spectra
  read 800,000 spectra
  read 1,000,000 spectra
  read 1,200,000 spectra
  read 1,400,000 spectra
  read 1,600,000 spectra
  read 1,800,000 spectra
  read 2,000,000 spectra
  read 2,200,000 spectra
  read 2,400,000 spectra
  read 2,539,608 spectra
reference: 2,296,616 spectra, 273,747 structures, 114s
  searched 0/250 molecules
  searched 50/250 molecules
  searched 100/250 molecules
  searched 150/250 molecules
  searched 200/250 molecules
[class-1-like (np-examples spectra removed, other libs kept)] molecules=250  top1=0.880  top5=0.976  top25=0.996
  searched 0/250 molecules
  searched 50/250 molecules
  searched 100/250 molecules
  searched 150/250 molecules
  searched 200/250 molecules
[hard (structure fully removed from reference)] molecules=250  top1=0.000  top5=0.000  top25=0.000
loading test
  read 1,213 spectra
  searched 0/400 molecules
  searched 50/400 molecules
  searched 100/400 molecules
  searched 150

In [ ]:
!pip install -q rdkit

In [ ]:
# Prepare COCONUT as a candidate-structure table for CASMI 2026 (class 2: known structure, no spectra).
#
# Run in a Kaggle notebook WITH INTERNET ON (dev session), then save /kaggle/working/coconut_candidates.parquet
# as a new Kaggle Dataset ("Save version" -> output -> "New dataset") and attach it to the submission notebook.
# The final scoring notebook probably has no internet, so the file must live in a dataset.
#
# Output columns: smiles, inchikey14, formula, mass (neutral monoisotopic)
#
#   pip install rdkit      (if not already available)
import io, os, sys, zipfile, urllib.request
import numpy as np, pandas as pd

URLS = [  # CC0 / CC-BY; Zenodo version first, official site second
    "https://zenodo.org/records/13897048/files/coconut-10-2024.csv.zip?download=1",
    "https://coconut.s3.uni-jena.de/prod/downloads/2026-11/coconut_csv_lite-11-2026.zip",
]
OUT = os.environ.get("OUT", "/kaggle/working/coconut_candidates.parquet")

def fetch():
    last = None
    for u in URLS:
        try:
            print("downloading", u, flush=True)
            data = urllib.request.urlopen(u, timeout=600).read()
            return zipfile.ZipFile(io.BytesIO(data))
        except Exception as e:
            print("  failed:", e); last = e
    raise last

def main():
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdMolDescriptors, Descriptors
    RDLogger.DisableLog("rdApp.*")

    zf = fetch()
    name = [n for n in zf.namelist() if n.lower().endswith(".csv")][0]
    print("csv:", name)
    df = pd.read_csv(zf.open(name), low_memory=False)
    print("columns:", list(df.columns)); print(df.head(2).T)

    cands = [c for c in df.columns if "smiles" in c.lower()]
    prefer = [c for c in cands if "canonical" in c.lower()] or cands
    if not prefer:
        sys.exit("no SMILES-like column; send me the printed column list")
    col = prefer[0]; print("using SMILES column:", col)
    smi = df[col].dropna().astype(str).drop_duplicates().tolist()
    print("raw structures:", len(smi))

    rows = []
    for i, s in enumerate(smi):
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        frags = Chem.GetMolFrags(m, asMols=True)          # keep the largest fragment
        m = max(frags, key=lambda x: x.GetNumHeavyAtoms())
        try:
            ik = Chem.MolToInchiKey(m)
        except Exception:
            continue
        if not ik:
            continue
        rows.append((Chem.MolToSmiles(m), ik.split("-")[0],
                     rdMolDescriptors.CalcMolFormula(m), Descriptors.ExactMolWt(m)))
        if i % 50000 == 0:
            print(f"  {i}/{len(smi)}", flush=True)
    out = pd.DataFrame(rows, columns=["smiles", "inchikey14", "formula", "mass"])
    out = out.drop_duplicates("inchikey14").sort_values("mass").reset_index(drop=True)
    out = out[(out.mass > 100) & (out.mass < 1500)]
    out.to_parquet(OUT)
    print(f"wrote {OUT}: {len(out):,} structures; mass range {out.mass.min():.1f}-{out.mass.max():.1f}")

if __name__ == "__main__":
    main()

downloading https://zenodo.org/records/13897048/files/coconut-10-2024.csv.zip?download=1
csv: coconut-10-2024.csv
columns: ['identifier', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key', 'name', 'iupac_name', 'total_atom_count', 'heavy_atom_count', 'molecular_weight', 'exact_molecular_weight', 'molecular_formula', 'alogp', 'topological_polar_surface_area', 'rotatable_bond_count', 'hydrogen_bond_acceptors', 'hydrogen_bond_donors', 'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski', 'lipinski_rule_of_five_violations', 'aromatic_rings_count', 'qed_drug_likeliness', 'formal_charge', 'fractioncsp3', 'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar', 'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework', 'np_likeness', 'chemical_class', 'chemical_sub_class', 'chemical_super_class', 'direct_parent_classification', 'np_classifier_pathway', 'np_classifier_superclass', 'np_classifier_class', 'np_classifier_is_glycoside']
              

In [ ]:
# Post-process coconut_candidates.parquet (output of prep_coconut.py):
#   1. neutralise charged forms (e.g. protonated amines [NH+]) so mass/InChIKey match the neutral structures in train
#   2. optionally strip stereochemistry (competition asks for 2D structures; inchikey14 ignores stereo anyway)
#   3. recompute mass / formula / inchikey14, drop duplicates, write back
# Needs RDKit (run in the same internet-enabled notebook as prep_coconut.py).
import os
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors, Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")
SRC = os.environ.get("SRC", "/kaggle/working/coconut_candidates.parquet")
DST = os.environ.get("DST", SRC)
STRIP_STEREO = int(os.environ.get("STRIP_STEREO", "1"))

un = rdMolStandardize.Uncharger()

def fix(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    if any(a.GetFormalCharge() != 0 for a in m.GetAtoms()):
        try:
            m = un.uncharge(m)
        except Exception:
            pass
    if STRIP_STEREO:
        Chem.RemoveStereochemistry(m)
    ik = Chem.MolToInchiKey(m)
    if not ik:
        return None
    return (Chem.MolToSmiles(m), ik.split("-")[0],
            rdMolDescriptors.CalcMolFormula(m), Descriptors.ExactMolWt(m))

df = pd.read_parquet(SRC)
charged = df["smiles"].str.contains(r"[+-]\]", regex=True)
print(f"{len(df):,} structures; {int(charged.sum()):,} look charged")
rows = []
for i, s in enumerate(df["smiles"]):
    r = fix(s)
    if r:
        rows.append(r)
    if i % 100000 == 0:
        print(f"  {i}/{len(df)}", flush=True)
out = (pd.DataFrame(rows, columns=["smiles", "inchikey14", "formula", "mass"])
         .drop_duplicates("inchikey14").sort_values("mass").reset_index(drop=True))
out.to_parquet(DST)
print(f"wrote {DST}: {len(out):,} structures; mass {out.mass.min():.1f}-{out.mass.max():.1f}")

466,587 structures; 16,950 look charged
  0/466587
  100000/466587
  200000/466587
  300000/466587
  400000/466587
wrote /kaggle/working/coconut_candidates.parquet: 466,489 structures; mass 100.0-1499.9


In [ ]:
import pandas as pd, pyarrow.parquet as pq

path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"
s = pq.read_table(path, columns=["normalized_smiles"]).to_pandas()["normalized_smiles"]
s = s.dropna().sample(100_000, random_state=0)
print("fraction with stereo marks:", s.str.contains(r"@|/|\\").mean())

fraction with stereo marks: 0.0


In [ ]:
# Spectrum -> molecular fingerprint model (for ranking COCONUT / PubChem candidates in class 2).
#
# Needs the reference cache written by baseline.py / baseline_v2.py:  ref_meta.parquet + ref_mat.npz in OUT_DIR
# (run baseline_v2.py once with QUICK unset in the same notebook; takes ~3 minutes).
# Needs: torch (GPU strongly recommended), rdkit.   pip install rdkit   (or the offline wheels dataset)
#
# Output: OUT_DIR/fp_model.pt  (+ prints a retrieval sanity check on held-out structures)
#
# Env: OUT_DIR, EPOCHS (default 4), BATCH (1024), HIDDEN (default 1024), MAX_MIN (time cap in minutes, default 120)
import os, time, numpy as np, pandas as pd, scipy.sparse as sp
import torch, torch.nn as nn
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")

OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
EPOCHS = int(os.environ.get("EPOCHS", "4"))
BATCH = int(os.environ.get("BATCH", "1024"))
MAX_MIN = float(os.environ.get("MAX_MIN", "120"))
NBITS = 2048
HIDDEN = int(os.environ.get("HIDDEN", "1024"))
BIN, MAX_MZ = 0.02, 1300.0            # must match baseline.py
DIM = int(MAX_MZ / BIN) + 4
F = 5                                  # re-bin 0.02 Da -> 0.1 Da
B = DIM // F + 1
IN = 2 * B + 2                         # fragments | losses | polarity | mass/1000
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

# ---- load cache ---------------------------------------------------------------------------------------
meta = pd.read_parquet(os.path.join(OUT_DIR, "ref_meta.parquet"))
mat = sp.load_npz(os.path.join(OUT_DIR, "ref_mat.npz")).tocsr()
print("reference:", mat.shape, "device:", dev, flush=True)
sid, uniq = pd.factorize(meta["inchikey14"])
first = pd.Series(np.arange(len(meta))).groupby(sid).first().to_numpy()
smiles = meta["normalized_smiles"].to_numpy()[first]
n_struct = len(uniq)

# ---- fingerprints per structure -----------------------------------------------------------------------
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
t0 = time.time()
FP = np.zeros((n_struct, NBITS), dtype=np.uint8)
valid = np.zeros(n_struct, bool)
for i, s in enumerate(smiles):
    m = Chem.MolFromSmiles(s)
    if m is not None:
        FP[i] = gen.GetFingerprintAsNumPy(m); valid[i] = True
    if i % 50000 == 0:
        print(f"  fingerprints {i}/{n_struct}", flush=True)
print(f"fingerprints done in {time.time()-t0:.0f}s, valid {valid.mean():.3f}")

# ---- split by structure (np-examples + 1% random held out) ---------------------------------------------
lib = meta["ingest_lib"].to_numpy()
hold = np.zeros(n_struct, bool)
hold[np.unique(sid[lib == "enveda-np-examples"])] = True
hold |= (np.random.rand(n_struct) < 0.01)
spec_train = np.flatnonzero(valid[sid] & ~hold[sid])
spec_hold = np.flatnonzero(valid[sid] & hold[sid])
print(f"train spectra {len(spec_train):,}  held-out spectra {len(spec_hold):,}  held-out structures {int(hold.sum()):,}")

mode = (meta["ionization_mode"] == "positive").to_numpy().astype(np.float32)
mass = meta["struct_mass"].to_numpy().astype(np.float32)

def make_batch(rows):
    sub = mat[rows]
    n = len(rows)
    r = np.repeat(np.arange(n), np.diff(sub.indptr))
    c = sub.indices.astype(np.int64)
    c = np.where(c < DIM, c // F, B + (c - DIM) // F)
    X = torch.zeros(n, IN, device=dev)
    X.index_put_((torch.from_numpy(r).to(dev), torch.from_numpy(c).to(dev)),
                 torch.from_numpy(sub.data).to(dev), accumulate=True)
    X[:, 2 * B] = torch.from_numpy(mode[rows]).to(dev)
    X[:, 2 * B + 1] = torch.from_numpy(mass[rows] / 1000.0).to(dev)
    return X

net = nn.Sequential(nn.Linear(IN, HIDDEN), nn.ReLU(), nn.Dropout(0.1),
                    nn.Linear(HIDDEN, HIDDEN), nn.ReLU(), nn.Dropout(0.1),
                    nn.Linear(HIDDEN, NBITS)).to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
steps_total = EPOCHS * (len(spec_train) // BATCH + 1)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps_total)
use_amp = dev == "cuda"
scaler = torch.amp.GradScaler(enabled=use_amp)
lossf = nn.BCEWithLogitsLoss()

@torch.no_grad()
def predict(rows, bs=2048):
    net.eval(); out = []
    for i in range(0, len(rows), bs):
        out.append(torch.sigmoid(net(make_batch(rows[i:i + bs]))).float().cpu())
    net.train()
    return torch.cat(out).numpy()

def retrieval_check(max_struct=3000):
    """Average predictions per held-out structure, rank all held-out true fingerprints by soft Tanimoto."""
    rows = spec_hold
    P = predict(rows)
    s_of = sid[rows]
    ids = np.unique(s_of)
    if len(ids) > max_struct:
        ids = np.random.choice(ids, max_struct, replace=False)
    keep = np.isin(s_of, ids)
    P, s_of = P[keep], s_of[keep]
    ids = np.unique(s_of)
    Pm = np.stack([P[s_of == i].mean(0) for i in ids])
    T = FP[ids].astype(np.float32)
    inter = Pm @ T.T
    tan = inter / (Pm.sum(1)[:, None] + T.sum(1)[None, :] - inter + 1e-9)
    true = np.diag(tan)
    rank = (tan > true[:, None]).sum(1)            # 0 = best
    print(f"  retrieval among {len(ids)} held-out structures: "
          f"top1={np.mean(rank < 1):.3f} top5={np.mean(rank < 5):.3f} top25={np.mean(rank < 25):.3f} "
          f"mean soft-Tanimoto(true)={true.mean():.3f}", flush=True)

t_start = time.time(); step = 0
for ep in range(EPOCHS):
    perm = np.random.permutation(spec_train)
    run = 0.0
    for bi in range(0, len(perm), BATCH):
        rows = np.sort(perm[bi:bi + BATCH])
        X = make_batch(rows)
        Y = torch.from_numpy(FP[sid[rows]]).float().to(dev)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = lossf(net(X), Y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if step < steps_total - 1:
            sched.step()
        step += 1; run += loss.item()
        if step % 200 == 0:
            print(f"ep {ep} step {step}/{steps_total} loss {run/200:.4f}  {(time.time()-t_start)/60:.1f} min", flush=True)
            run = 0.0
        if (time.time() - t_start) / 60 > MAX_MIN:
            break
    print(f"== epoch {ep} done", flush=True)
    retrieval_check()
    torch.save({"state": net.state_dict(), "IN": IN, "B": B, "F": F, "DIM": DIM, "NBITS": NBITS, "H": HIDDEN},
               os.path.join(OUT_DIR, "fp_model.pt"))
    if (time.time() - t_start) / 60 > MAX_MIN:
        print("time cap reached"); break
print("saved", os.path.join(OUT_DIR, "fp_model.pt"))

reference: (2296616, 130008) device: cuda
  fingerprints 0/273747
  fingerprints 50000/273747
  fingerprints 100000/273747
  fingerprints 150000/273747
  fingerprints 200000/273747
  fingerprints 250000/273747
fingerprints done in 67s, valid 1.000
train spectra 2,230,388  held-out spectra 66,228  held-out structures 3,032
ep 0 step 200/8716 loss 0.4569  0.1 min
ep 0 step 400/8716 loss 0.0905  0.2 min
ep 0 step 600/8716 loss 0.0846  0.2 min
ep 0 step 800/8716 loss 0.0811  0.3 min
ep 0 step 1000/8716 loss 0.0798  0.4 min
ep 0 step 1200/8716 loss 0.0767  0.5 min
ep 0 step 1400/8716 loss 0.0752  0.6 min
ep 0 step 1600/8716 loss 0.0744  0.7 min
ep 0 step 1800/8716 loss 0.0738  0.7 min
ep 0 step 2000/8716 loss 0.0731  0.8 min
== epoch 0 done
  retrieval among 3000 held-out structures: top1=0.014 top5=0.048 top25=0.137 mean soft-Tanimoto(true)=0.162
ep 1 step 2200/8716 loss 0.0076  0.9 min
ep 1 step 2400/8716 loss 0.0719  1.0 min
ep 1 step 2600/8716 loss 0.0713  1.1 min
ep 1 step 2800/8716 lo

In [ ]:
import os
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/__notebook__.ipynb
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/__output__.json
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/__results__.html
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/custom.css
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/ref_mat.npz
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/ref_meta.parquet
/kaggle/input/notebooks/sarahnjunge/coconut-candidates/submission.csv
